# 002 — Fock smooth-Earth diffraction (globe), Stollberg

ITU-R P.526-16 §3 Fock first-term residue series. Mirrors ITU sheet
columns V through BP for row 3.

This is the **globe (GE) model** — exponential field-strength decay
past the geometric horizon caused by the wave bending around the
sphere of radius 6 371 km. Every Knickebein path past the radio
horizon enters the residue-series shadow zone, and the loss in that
zone grows linearly with distance.

The cross beam (Stollberg) runs from Stollberg over sea (sigma = 5 S/m, eps_r = 70, beta = 0.81) to a
6 000 m He 111 receiver. Change `target` in the next cell to any
name from `TARGETS` and re-run.


In [1]:
import math, cmath, pandas as pd
from IPython.display import Markdown, display
import common as c

station = 'Stollberg'
target  = 'Beeston'      # change me

s = c.STATIONS[station]
t = c.TARGETS[target]
f_MHz = s['freq_MHz']
ground = 'sea' if target.startswith('TF') else s['ground']


## Geometry on a globe

Great-circle distance via haversine (ITU sheet cell `O2` / `O3`):
$$d = 2 R_\oplus \arcsin\sqrt{\sin^2\!\tfrac{\Delta\varphi}{2} + \cos\varphi_1 \cos\varphi_2 \sin^2\!\tfrac{\Delta\lambda}{2}}$$

Effective Earth radius (4/3 model, ITU-R P.453):
$$a_e = \tfrac{4}{3} R_\oplus = 8\,494\,667\ \text{m}$$

Radio horizon, each end:
$$d_{\mathrm{LoS},\mathrm{tx}} = \sqrt{2 a_e h_{tx}}, \qquad d_{\mathrm{LoS},\mathrm{rx}} = \sqrt{2 a_e h_{rx}}$$

If $d < d_{\mathrm{LoS},\mathrm{tx}} + d_{\mathrm{LoS},\mathrm{rx}}$ the path is line-of-sight and Fock returns zero diffraction loss. Otherwise the path is in the diffraction shadow zone.


In [2]:
d_m  = c.great_circle_m(s['lat_deg'], s['lon_deg'], t['lat'], t['lon'])
h_tx = s['h_tx_m']
h_rx = t['rx_alt_m']
d_los_tx = math.sqrt(2*c.R_EFFECTIVE*h_tx)
d_los_rx = math.sqrt(2*c.R_EFFECTIVE*h_rx)
d_los    = d_los_tx + d_los_rx
in_shadow = d_m > d_los
display(Markdown(f'''
- $d$ = **{d_m/1000:.2f} km**
- $h_{{tx}}$ = {h_tx} m, $h_{{rx}}$ = {h_rx} m
- $d_{{LoS,tx}}$ = {d_los_tx/1000:.2f} km
- $d_{{LoS,rx}}$ = {d_los_rx/1000:.2f} km
- $d_{{LoS}}$ total = **{d_los/1000:.2f} km**
- shadow zone? **{in_shadow}**, shadow length = {(d_m-d_los)/1000 if in_shadow else 0:.2f} km
'''))



- $d$ = **693.50 km**
- $h_{tx}$ = 72 m, $h_{rx}$ = 6000 m
- $d_{LoS,tx}$ = 34.97 km
- $d_{LoS,rx}$ = 319.27 km
- $d_{LoS}$ total = **354.25 km**
- shadow zone? **True**, shadow length = 339.25 km


## $\beta$ polarisation parameter (ITU-R P.526-16 Eqs. 16, 16a)

The Fock $\beta$ is unity for horizontal polarisation at any frequency,
unity for vertical polarisation over land above 20 MHz, and unity for
vertical polarisation over sea above 300 MHz. Otherwise it is computed
from the surface admittance $K$:

$$K^2 \approx \frac{6.89\,\sigma}{k^{2/3}\,f_{MHz}^{5/3}} \qquad \text{(Eq. 16a)}$$

$$\beta = \frac{1 + 1.6 K^2 + 0.67 K^4}{1 + 4.5 K^2 + 1.53 K^4} \qquad \text{(Eq. 16)}$$

Knickebein at 31.5 MHz vertical pol falls into the rule where:

- over **land** (above the 20 MHz cut), $\beta \to 1$;
- over **sea**  (below the 300 MHz cut), $\beta$ is taken from Eq. 16
  using $K$ from Eq. 16a. For sea $\sigma = 5$ S/m the result is
  $\beta \approx 0.81$, and $K$ propagates into the lower-bound clamp on
  $G(Y)$ via Eq. 18.


In [3]:
beta, K_floor = c.p526_beta_and_K(ground, s['pol'], f_MHz, c.K_REFRAC)
sigma = c.GROUND[ground]['sigma']
K2 = 6.89*sigma / (c.K_REFRAC**(2/3) * f_MHz**(5/3))
K  = math.sqrt(K2)
beta_eq16 = (1 + 1.6*K2 + 0.67*K2**2) / (1 + 4.5*K2 + 1.53*K2**2)
display(Markdown(f'''
- ground = **{ground}**, sigma = {sigma} S/m
- $K^2$ = {K2:.4g}, $K$ = {K:.4f}
- $\beta$ from Eq. 16 = {beta_eq16:.4f}
- $\beta$ final = **{beta:.4f}** (over-ride to 1 above frequency cut)
- $K$ used in $G(Y)$ floor = {K_floor:.4f}
'''))



- ground = **sea**, sigma = 5.0 S/m
- $K^2$ = 0.09051, $K$ = 0.3009
- $eta$ from Eq. 16 = 0.8102
- $eta$ final = **0.8102** (over-ride to 1 above frequency cut)
- $K$ used in $G(Y)$ floor = 0.3009


## Normalised distance and heights

$$X = \beta \left(\frac{\pi}{\lambda a_e^2}\right)^{1/3} d \qquad \text{(Eq. 13)}$$
$$Y_{1,2} = 2 \beta \left(\frac{\pi^2}{\lambda^2 a_e}\right)^{1/3} h_{tx,rx} \qquad \text{(Eq. 13)}$$


In [4]:
X  = c.p526_X(d_m, beta, f_MHz)
Y1 = c.p526_Y(h_tx, beta, f_MHz)
Y2 = c.p526_Y(h_rx, beta, f_MHz)
display(Markdown(f'''
- $X$ = **{X:.3f}**  (long-path branch if X >= 1.6)
- $Y_1$ (TX) = **{Y1:.3f}**
- $Y_2$ (RX) = **{Y2:.3f}**
- long-path branch? **{X >= 1.6}**
'''))



- $X$ = **9.327**  (long-path branch if X >= 1.6)
- $Y_1$ (TX) = **0.273**
- $Y_2$ (RX) = **22.758**
- long-path branch? **True**


## Distance term $F(X)$ (Eq. 14 / 15)

$$F(X) = \begin{cases}
11 + 10\log_{10} X - 17.6\,X & X \ge 1.6 \\
-20\log_{10} X - 5.6488\,X^{1.425} & X < 1.6
\end{cases}$$

## Height-gain term $G(Y)$ (Eq. 17, with Eq. 18 floor)

$$G(Y) = \begin{cases}
17.6\sqrt{B - 1.1} - 5\log_{10}(B - 1.1) - 8 & B > 2 \\
20\log_{10}\bigl(B + 0.1 B^3\bigr) & B \le 2
\end{cases}, \quad B = \beta Y$$

$$G(Y) \ge 2 + 20 \log_{10} K \quad \text{(Eq. 18 lower bound, when sea)}$$

Field strength relative to free space:
$$E/E_0\ \text{(dB)} = F(X) + G(Y_1) + G(Y_2)$$

Diffraction loss is the negative of this, clipped at zero inside the
line of sight.


In [5]:
F_X  = c.F_of_X(X)
B1, B2 = beta*Y1, beta*Y2
G_Y1 = c.G_of_Y(B1, K_floor)
G_Y2 = c.G_of_Y(B2, K_floor)
E_over_E0 = F_X + G_Y1 + G_Y2
L_diff = c.fock_diff_loss_dB(d_m, h_tx, h_rx, f_MHz, ground, s['pol'])
display(Markdown(f'''
- $F(X)$  = **{F_X:.3f} dB**
- $G(Y_1)$ = {G_Y1:.3f} dB
- $G(Y_2)$ = {G_Y2:.3f} dB
- $E/E_0$  = **{E_over_E0:.3f} dB**
- diffraction loss applied = **{L_diff:.3f} dB**
'''))



- $F(X)$  = **-143.455 dB**
- $G(Y_1)$ = -8.433 dB
- $G(Y_2)$ = 59.089 dB
- $E/E_0$  = **-92.799 dB**
- diffraction loss applied = **92.799 dB**


## Link budget (ITU sheet cols AS-BP)

$$\text{FSPL} = 20 \log_{10}\!\frac{4\pi d}{\lambda}, \qquad G_{tx} = 10\log_{10}\!\frac{4\pi A}{\lambda^2}$$

$$P_{rx} = P_{tx} + G_{tx} + G_{rx} - \text{FSPL} - L_{\text{diffraction}}$$

Galactic noise (ITU-R P.372-16 Eq. 14, $f < 100$ MHz):
$$F_a = 52 - 23 \log_{10} f_{MHz}$$

Noise floor at the 50 ohm input:
$$N = 10\log_{10}(k T_{sys} B) + \max(NF, F_a)$$

Crossover loss (5° squint of the 99 m aperture):
$$L_{\text{cross}} = 20\log_{10}\!\left|\frac{\sin(\pi W \sin\theta/\lambda)}{\pi W \sin\theta/\lambda}\right|$$

The pilot flies the equisignal corridor, not the sub-beam peak, so the
equisignal SNR is $\text{SNR}_{\text{peak}} + L_{\text{cross}}$ (the
loss is negative).


In [6]:
r = c.link_budget(station, target, model='fock')
pd.DataFrame({
    'Quantity':[
        'FSPL (dB)',
        'G_tx (dBi)',
        'P_tx (dBW)',
        'P_rx (dBW)',
        'Noise floor (dBW)',
        'SNR peak (dB)',
        'Crossover at 5 deg (dB)',
        'SNR equisignal (dB)',
        'V_eq at 50 ohm (uV)',
        'V_noise at 50 ohm (uV)',
    ],
    'Value':[
        r['FSPL_dB'], r['G_tx_dBi'], r['P_tx_dBW'],
        r['P_rx_dBW'], r['N_dBW'], r['SNR_peak_dB'],
        r['crossover_dB'], r['SNR_eq_dB'],
        r['V_eq_uV'], r['V_noise_uV'],
    ],
}).style.format({'Value': '{:.4g}'})


,Quantity,Value
0,FSPL (dB),119.2
1,G_tx (dBi),26
2,P_tx (dBW),34.77
3,P_rx (dBW),-151.3
4,Noise floor (dBW),-159.4
5,SNR peak (dB),8.186
6,Crossover at 5 deg (dB),-19.87
7,SNR equisignal (dB),-11.68
8,V_eq at 50 ohm (uV),0.01966
9,V_noise at 50 ohm (uV),0.07545


## Verdict against the +10 dB detection floor

The 0.079 uV physics noise floor at the 50 ohm input is the bare
detection threshold once the FuBl 2 / EBL 3 receiver AGC is taken
into account. The +10 dB margin is the standard bare-RF detection
floor above noise.


In [7]:
verdict = ('PASS' if r['SNR_eq_dB'] >= 10
           else 'MARGINAL' if r['SNR_eq_dB'] >= 0
           else 'FAIL')
display(Markdown(f'''
- SNR equisignal = **{r['SNR_eq_dB']:+.2f} dB**
- V equisignal   = **{r['V_eq_uV']:.4g} uV**  (noise floor {r['V_noise_uV']:.4g} uV)
- verdict = **{verdict}**
'''))



- SNR equisignal = **-11.68 dB**
- V equisignal   = **0.01966 uV**  (noise floor 0.07545 uV)
- verdict = **FAIL**


## Sweep all confirmed Stollberg paths

This is the table the spreadsheet's Main / ITU rows produce when the
target dropdown is cycled.


In [8]:
rows = []
for tgt in ['Beeston', 'Derby', 'Birmingham', 'Liverpool', 'TF 400 km', 'TF 500 km', 'TF 700 km', 'TF 800 km', 'TF 1000 km']:
    rr = c.link_budget(station, tgt, model='fock')
    v = ('PASS' if rr['SNR_eq_dB'] >= 10
         else 'MARGINAL' if rr['SNR_eq_dB'] >= 0
         else 'FAIL')
    rows.append((tgt, round(rr['d_km'],1), rr['ground'],
                 round(rr['diffraction_loss_dB'],2),
                 round(rr['SNR_peak_dB'],2),
                 round(rr['SNR_eq_dB'],2),
                 round(rr['V_eq_uV'],4), v))
pd.DataFrame(rows, columns=['target','d_km','ground','L_diff_dB',
                            'SNRpeak_dB','SNReq_dB','V_eq_uV','verdict'])


,target,d_km,ground,L_diff_dB,SNRpeak_dB,SNReq_dB,V_eq_uV,verdict
0,Beeston,693.5,sea,92.80,8.19,-11.68,0.0197,FAIL
1,Derby,710.1,sea,96.62,4.16,-15.71,0.0124,FAIL
2,Birmingham,753.8,sea,106.71,-6.44,-26.31,0.0036,FAIL
3,Liverpool,790.6,sea,115.23,-15.38,-35.25,0.0013,FAIL
4,TF 400 km,413.7,sea,42.26,63.22,43.35,11.0942,PASS
5,TF 500 km,504.7,sea,62.94,40.80,20.94,0.8403,PASS
6,TF 700 km,702.6,sea,108.34,-7.47,-27.34,0.0032,FAIL
7,TF 800 km,805.3,sea,132.07,-32.39,-52.25,0.0002,FAIL
8,TF 1000 km,1000.1,sea,177.23,-79.42,-99.29,0.0000,FAIL


## Squint Sandbox — edit and observe

The Squint Sandbox sheet in the workbook is the calibration cell.
Edit `squint_deg`, `W_m`, `H_m` below and re-run to see how the
equisignal corridor width and the equisignal SNR move.

The defaults (`squint = 5 deg, W = 99 m, H = 20 m`) reproduce the
400 to 500 yard equisignal corridor at Spalding measured by R/T flight
F/Lt Bufton on 21 June 1940.


In [9]:
squint_deg = c.SANDBOX['squint_deg']   # default 5
W_m        = c.SANDBOX['W_m']          # default 99
H_m        = c.SANDBOX['H_m']          # default 20

lam = c.freq_to_wavelen(f_MHz)
u  = math.pi*W_m*math.sin(math.radians(squint_deg))/lam
L_cross = 20*math.log10(abs(math.sin(u)/u))
G_sub_dBi  = 10*math.log10(4*math.pi*W_m*H_m / lam**2)
width_m = c.equisignal_corridor_width_m(d_m, W_m, squint_deg, f_MHz)
SNR_eq  = r['SNR_peak_dB'] + L_cross
display(Markdown(f'''
- u = pi W sin(theta)/lambda = {u:.4f}
- crossover loss = **{L_cross:.3f} dB**
- aperture directivity (W={W_m} m, H={H_m} m) = **{G_sub_dBi:.2f} dBi**
- equisignal corridor width at d = {d_m/1000:.1f} km
  -> **{width_m:.1f} m** = **{width_m/0.9144:.1f} yards**
- SNR equisignal (using peak from above) = **{SNR_eq:+.2f} dB**
'''))



- u = pi W sin(theta)/lambda = 2.8482
- crossover loss = **-19.867 dB**
- aperture directivity (W=99.0 m, H=20.0 m) = **24.39 dBi**
- equisignal corridor width at d = 693.5 km
  -> **669.1 m** = **731.8 yards**
- SNR equisignal (using peak from above) = **-11.68 dB**
